# Binary Search — Cheat Sheet

**Companion to:** Sliding Window & Two Pointers Study Guide

**What this adds:** Binary search was referenced in the study guides as a tool (e.g. LIS O(n log n) variant) but never taught as a standalone topic. This cheat sheet covers the loop variant decision, the three templates, and the full class of binary search on answer problems.

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| "sorted array", "search for target" | Classic binary search |
| "find first / last position", "leftmost / rightmost" | Binary search on boundary |
| "minimum/maximum value that satisfies a condition" | Binary search on answer |
| "kth smallest in sorted matrix / two sorted arrays" | Binary search on value range |
| O(log n) hint in problem constraints | Binary search |

---
## Patterns at a Glance

| Pattern | Condition | Loop variant | Returns | Problems |
| :--- | :--- | :--- | :--- | :--- |
| Classic | Exact match | `left <= right` | Index or -1 | 704 |
| Left boundary | First position satisfying condition | `left < right` | `left` | 34, 35, 300 |
| Right boundary | Last position satisfying condition | `left < right` | `left - 1` | 34 |
| Search on answer | Min/max value satisfying a condition | `left < right` | `left` | 875, 1011, 410 |

---
# Part 0 — Quick Recap

Binary search eliminates half the search space at each step by comparing the midpoint to a target. It requires the search space to be **monotonic** — either sorted values, or a condition that transitions cleanly from False to True (or True to False) with no back-and-forth.

The entire intellectual content of binary search is the **loop variant decision** — which of the three templates you use determines whether your final `left` pointer lands on the correct boundary. Getting this wrong by one produces subtle off-by-one bugs that are hard to debug under pressure.

**Closest neighbor to distinguish from:**
- Binary Search vs Two Pointers: use Binary Search when the array is sorted and you need O(log n); use Two Pointers when you need O(n) and the array may not be sorted
- Binary Search on value vs Binary Search on index: "search on answer" problems search over a **value range**, not an array index — the midpoint is a candidate answer, not an array position

---
# Part 1 — The Three Templates

## Core idea

The three templates differ in only two places: the loop condition and the mid assignment. Everything else is identical. Choose based on what you want `left` to point to when the loop terminates.

In [ ]:
# ── Template 1: Classic — exact match (left <= right) ────────────────────────
# Use when: searching for an exact target value
# Terminates when: left > right (search space exhausted)
# Returns: index of target or -1
def binary_search_classic(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:                # = allows single-element search space
        mid = left + (right - left) // 2  # avoids overflow vs (left+right)//2
        if nums[mid] == target:   return mid
        elif nums[mid] < target:  left  = mid + 1
        else:                     right = mid - 1
    return -1


# ── Template 2: Left boundary — first position satisfying condition ───────────
# Use when: finding leftmost index where nums[mid] >= target (or any condition)
# Terminates when: left == right, pointing to the first valid position
# Returns: left (insertion point / first match)
def binary_search_left(nums, target):
    left, right = 0, len(nums)          # right = len(nums) to handle insert-at-end
    while left < right:                 # no = — loop ends when left == right
        mid = left + (right - left) // 2
        if nums[mid] < target:  left  = mid + 1
        else:                   right = mid      # mid stays in range — don't exclude it
    return left                         # left == right == first position >= target


# ── Template 3: Right boundary — last position satisfying condition ───────────
# Use when: finding rightmost index where nums[mid] <= target
# Terminates when: left == right, one past the last valid position
# Returns: left - 1 (last match)
def binary_search_right(nums, target):
    left, right = 0, len(nums)
    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] <= target: left  = mid + 1  # mid is valid — move past it
        else:                   right = mid
    return left - 1                     # left - 1 = last position <= target


# ── Python built-in: bisect ───────────────────────────────────────────────────
import bisect
bisect.bisect_left(nums, target)    # = Template 2: first position >= target
bisect.bisect_right(nums, target)   # = Template 3 + 1: first position > target

## Variants

- **Rotated sorted array (LC 33):** determine which half is sorted, then check if target is in that half
- **2D sorted matrix (LC 74):** treat as 1D array — map `mid` to `(mid // cols, mid % cols)`
- **Unknown size / infinite array:** double `right` until `nums[right] >= target`, then binary search

## Common mistakes

- Using `left <= right` when you need a boundary — causes infinite loop when `left == right` and `mid == left` keeps setting `right = mid`
- Writing `mid = (left + right) // 2` — overflows in languages with fixed integers; always use `left + (right - left) // 2`
- Setting `right = len(nums) - 1` in Template 2/3 — misses the insert-at-end case; use `right = len(nums)`

**Practice problems:**

| Problem | Template | Notes |
| :--- | :--- | :--- |
| LC 704 — Binary Search | Classic | Baseline — use as warmup |
| LC 35 — Search Insert Position | Left boundary | `bisect_left` equivalent |
| LC 34 — First and Last Position | Left + Right boundary | Run both templates independently |
| LC 33 — Search in Rotated Sorted Array | Classic + half-sorted logic | Identify sorted half first |

---
# Part 2 — Binary Search on Answer

## Core idea

Instead of searching for a value in an array, you search over a **range of candidate answers**. At each midpoint you ask: "is this candidate answer feasible?" The search space is the range of possible answer values — always monotonic (if `x` is feasible, so is `x+1` for minimization problems).

**The key shift in thinking:** `mid` is not an array index — it is a candidate answer value. You write a helper `canAchieve(mid)` that checks feasibility, then binary search on that.

In [ ]:
# ── Template: Binary Search on Answer ────────────────────────────────────────
# Search space: [min_possible_answer, max_possible_answer]
# Goal: find minimum value where condition(mid) is True
def binary_search_on_answer(data):
    left, right = MIN_ANSWER, MAX_ANSWER
    while left < right:
        mid = left + (right - left) // 2
        if feasible(mid, data):     # can we achieve the answer with value = mid?
            right = mid             # mid works — try smaller
        else:
            left = mid + 1          # mid doesn't work — need larger
    return left                     # minimum feasible answer


# Example: LC 875 — Koko Eating Bananas
# Find minimum eating speed k such that Koko finishes all bananas in h hours
import math
def minEatingSpeed(piles, h):
    def feasible(speed):
        return sum(math.ceil(p / speed) for p in piles) <= h

    left, right = 1, max(piles)     # search space: 1 banana/hr to max pile size
    while left < right:
        mid = left + (right - left) // 2
        if feasible(mid): right = mid
        else:             left  = mid + 1
    return left


# Example: LC 1011 — Capacity to Ship Packages Within D Days
# Find minimum ship capacity to ship all packages within d days
def shipWithinDays(weights, days):
    def feasible(capacity):
        needed, cur = 1, 0
        for w in weights:
            if cur + w > capacity:
                needed += 1
                cur = 0
            cur += w
        return needed <= days

    left, right = max(weights), sum(weights)  # min = heaviest package, max = all in one day
    while left < right:
        mid = left + (right - left) // 2
        if feasible(mid): right = mid
        else:             left  = mid + 1
    return left

## Variants

- **Maximize instead of minimize:** flip `feasible` condition and swap `left`/`right` assignment
- **Search space is float range:** use `while right - left > 1e-6` instead of `while left < right`
- **Feasibility check is expensive:** make sure it runs in O(n) or better — the full solution is O(n log(range))

## Common mistakes

- Setting search space bounds wrong — `left` should be the smallest possible answer (not 0), `right` should be the largest (not `len(array)`)
- Writing `feasible` that checks `>` when it should check `>=` — off-by-one in the condition flips the entire search direction
- Forgetting that for maximization problems the assignment flips: `if feasible(mid): left = mid + 1` instead of `right = mid`

**Practice problems:**

| Problem | Search space | Feasibility check | Notes |
| :--- | :--- | :--- | :--- |
| LC 875 — Koko Eating Bananas | `[1, max(piles)]` | Total hours <= h | Classic minimize template |
| LC 1011 — Ship Packages in D Days | `[max(w), sum(w)]` | Days needed <= d | Same structure as 875 |
| LC 410 — Split Array Largest Sum | `[max(nums), sum(nums)]` | Pieces needed <= m | Minimize the maximum subarray sum |
| LC 4 — Median of Two Sorted Arrays | Value range | Count of elements <= mid | Hardest — binary search on value, not index |

---
# Decision Guide

```
Is the search space monotonic (sorted array or condition that flips once)?
└── Yes → Binary Search
          │
          ├── Searching for an exact value in a sorted array?
          │   └── Classic template (left <= right)
          │       └── Rotated sorted array → identify sorted half first (LC 33)
          │
          ├── Finding the first or last position of a value?
          │   ├── First position (leftmost)   → Left boundary (left < right, right = mid)
          │   └── Last position (rightmost)   → Right boundary (left < right, left = mid+1)
          │
          └── Finding the minimum/maximum value that satisfies a condition?
              └── Binary Search on Answer
                  ├── Define search space: left = min possible, right = max possible
                  ├── Write feasible(mid) → bool
                  ├── Minimize → if feasible: right = mid, else left = mid + 1
                  └── Maximize → if feasible: left = mid + 1, else right = mid
```